# Домашняя работа: анализ данных и feature engineering

Работаем с **Adult**. [Скачайте данные в задаче A](https://contest.yandex.ru/contest/101948/problems/A/) и положите CSV в папку `data` рядом с ноутбуком.

**Как сдавать A–K:** один `.py` на задачу, с нужными функциями, импортами и помощниками. Без запуска кода, чтения файлов, `print` и `assert`. Компилятор — **(make) Python 3.12 (Data Science)**. Полные требования и пограничные случаи — по ссылкам в заголовках задач.

Используйте готовые train/validation и `random_state=42`. Статистики считайте только по train; при применении не пересчитывайте. Исходные данные не меняйте, сохраняйте индекс и порядок строк. Квантили считайте с линейной интерполяцией.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from IPython.display import display

# Для случайных экспериментов с NumPy.
np.random.seed(42)

## Adult

Теперь вы пройдете путь от анализа признаков до постепенного улучшения качества модели путем работы с ними самостоятельно. В Adult - датасете для бинарной классификации - одна строка описывает человека. Попробуем предсказать `income_gt_50k`: 1 означает годовой доход выше 50 тысяч долларов, 0 — не выше. Какие признаки, на ваш взгляд, могут помочь?

| Признак | Смысл |
|---|---|
| `age` | Возраст, годы |
| `workclass` | Тип занятости или работодателя |
| `education`, `education_num` | Уровень образования: название и его порядковый номер |
| `occupation` | Профессиональная группа |
| `relationship` | Положение человека в домохозяйстве |
| `capital_gain`, `capital_loss` | Доходы и потери от капитала, доллары |
| `hours_per_week` | Число рабочих часов в неделю |
| `native_country` | Страна происхождения |

Для оценки возьмём **ROC-AUC**: она показывает, насколько хорошо модель ставит людей с доходом >50K выше остальных по предсказанной вероятности. Чем больше, тем лучше; 0.5 соответствует случайному ранжированию, 1 — идеальному.

In [ ]:
adult_train = pd.read_csv('data/adult_train.csv')
adult_valid = pd.read_csv('data/adult_valid.csv')
X_adult, y_adult = adult_train.drop(columns='income_gt_50k'), adult_train['income_gt_50k']
X_adult_valid, y_adult_valid = adult_valid.drop(columns='income_gt_50k'), adult_valid['income_gt_50k']
adult_train.head()

### [A. Первичный анализ](https://contest.yandex.ru/contest/101948/problems/A/)

Отправьте три функции:

- `dataset_profile(X, y)` → `(n_rows, n_features, n_numeric, n_categorical, positive_rate)`. Target не входит в X; числовые колонки — dtype `number`, остальные — категории; `positive_rate` — среднее y.
- `missing_report(X)` → DataFrame: индекс — колонки X, поля — `missing_count`, `missing_share`. Доля пропусков — от всех строк.
- `numeric_summary(X, columns)` → DataFrame: индекс — `columns`, поля — `count`, `mean`, `median`, `q25`, `q75`, `p95`, `zero_share`. Пропуски не учитывайте; доля нулей — среди известных значений.

Сохраняйте указанный порядок. Для пустых данных: счётчики — 0, остальные статистики — NaN.


In [ ]:
import numpy as np
import pandas as pd

def dataset_profile(X, y):
    # TODO: верните n_rows, n_features, n_numeric, n_categorical, positive_rate.
    return None, None, None, None, None

def missing_report(X):
    # TODO: замените None результатами вычислений для каждой колонки X.
    return pd.DataFrame({
        'missing_count': None,
        'missing_share': None,
    }, index=X.columns)

def numeric_summary(X, columns):
    # TODO: замените None результатами вычислений для каждой колонки из columns.
    return pd.DataFrame({
        'count': None,
        'mean': None,
        'median': None,
        'q25': None,
        'q75': None,
        'p95': None,
        'zero_share': None,
    }, index=columns)

In [ ]:
sample = pd.DataFrame({'value': [0., 2., 4., np.nan], 'group': ['a', 'b', 'a', None]})
labels = pd.Series([0, 1, 1, 0])
assert dataset_profile(sample, labels) == (4, 2, 1, 1, 0.5)
assert missing_report(sample)['missing_count'].tolist() == [1, 1]
assert np.allclose(numeric_summary(sample, ['value']).loc['value'], [3, 2, 2, 1, 3, 3.8, 1/3])

In [ ]:
display(dataset_profile(X_adult, y_adult))
display(missing_report(X_adult))
display(numeric_summary(X_adult, ['age', 'hours_per_week', 'capital_gain']))
X_adult[['age', 'hours_per_week', 'capital_gain']].hist(bins=30, figsize=(10, 3))
plt.tight_layout()
plt.show()
# Запишите: где есть пропуски, где длинный хвост, где много нулей.

### [B. Групповая сводка](https://contest.yandex.ru/contest/101948/problems/B/)

`group_summary(X, y, groups)` → DataFrame с индексом `group` в порядке первого появления групп:

- `n` — число строк;
- `median_hours` — медиана `hours_per_week` без пропусков;
- `positive_rate` — среднее y.

`groups` — Series с индексом X. Пропущенная группа — `__MISSING__`; неиспользованные категории не включайте.


In [ ]:
import pandas as pd

def group_summary(X, y, groups):
    # TODO: замените None статистиками, а [] — группами в порядке первого появления.
    return pd.DataFrame({
        'n': None,
        'median_hours': None,
        'positive_rate': None,
    }, index=pd.Index([], name='group'))

In [ ]:
sample = pd.DataFrame({'hours_per_week': [20., 40., np.nan]})
result = group_summary(sample, pd.Series([0, 1, 1]), pd.Series(['a', 'b', 'a']))
assert result.loc['a'].tolist() == [2, 20, 0.5]

In [ ]:
education_summary = group_summary(X_adult, y_adult, X_adult['education'])
display(education_summary.sort_values('positive_rate', ascending=False))
age_groups = pd.cut(X_adult['age'], [0, 25, 35, 45, 55, 65, np.inf], right=False)
display(group_summary(X_adult, y_adult, age_groups))
# Запишите одну закономерность и одно ограничение такой интерпретации.

### Вернёмся к пропускам — исследование

Теперь мы лучше знаем данные. По сводке из A выберите категориальные колонки с пропусками и примените способы с семинара: константу, моду и константу с флагом.

Посмотрите, как изменились частоты профессий. Какой способ вы оставите и какую информацию он сохраняет? Запишите своё решение — оно пригодится при сборке модели. Числовые признаки Adult полные, дополнительно скрывать значения здесь не нужно.

Ниже повторены функции для заполнения категорий. Заполнение константой уже готово.
Для остальных перенесите свои решения с семинара или реализуйте их здесь:
`fit_category_modes` сохраняет моды train (при отсутствии известных значений —
`__MISSING__`, при нескольких модах — первую из `Series.mode()`),
`fill_categories_mode` применяет готовые моды, а `fill_categories_with_flags`
заполняет пропуски `__MISSING__` и добавляет бинарные колонки `<колонка>_missing`.
Все функции возвращают новые данные, сохраняя исходные таблицы.

In [ ]:
def fill_categories_constant(X, columns, value='__MISSING__'):
    result = X.copy()
    result[columns] = result[columns].astype(object).fillna(value)
    return result


def fit_category_modes(X, columns):
    # TODO
    return {col: None for col in columns}


def fill_categories_mode(X, modes):
    # TODO: примените готовые моды к копии таблицы.
    raise NotImplementedError


def fill_categories_with_flags(X, columns):
    # TODO: заполните константой и добавьте флаги исходных пропусков.
    raise NotImplementedError

In [ ]:
# TODO: сравните три способа на occupation и выберите подходящий.
raise NotImplementedError

### [C. Выбросы по IQR](https://contest.yandex.ru/contest/101948/problems/C/)

- `fit_iqr_bounds(X, columns, factor=1.5)` → DataFrame с индексом `columns` и полями `lower`, `upper`: `Q₁ − factor·IQR`, `Q₃ + factor·IQR`, где `IQR = Q₃ − Q₁`.
- `outlier_mask(X, bounds)` → булевый DataFrame по колонкам из `bounds`: True только строго за границами; для пропусков — False.

Для полностью пустой train-колонки: границы NaN, маска False. При IQR=0 используйте ту же формулу. Неконечный или отрицательный `factor` → `ValueError`.


In [ ]:
import numpy as np
import pandas as pd

def fit_iqr_bounds(X, columns, factor=1.5):
    # TODO
    raise NotImplementedError()

def outlier_mask(X, bounds):
    # TODO
    raise NotImplementedError()

In [ ]:
bounds = fit_iqr_bounds(pd.DataFrame({'value': [0., 1., 2., 3., 4.]}), ['value'])
assert np.allclose(bounds.loc['value'], [-2., 6.])
assert outlier_mask(pd.DataFrame({'value': [-2., 7., np.nan]}), bounds)['value'].tolist() == [False, True, False]

### [D. Ограничение крайних значений](https://contest.yandex.ru/contest/101948/problems/D/)

- `fit_clip_bounds(X, columns, lower=0.01, upper=0.99)` → DataFrame с индексом `columns` и полями `lower`, `upper` — соответствующими квантилями train.
- `apply_clip(X, bounds)` → копия X: значения ниже/выше порога заменены этим порогом. Пропуски и остальные колонки сохраняются; граница NaN не ограничивает соответствующую сторону.

Если не выполнено `0 <= lower <= upper <= 1`, вызовите `ValueError`.


In [ ]:
import pandas as pd

def fit_clip_bounds(X, columns, lower=0.01, upper=0.99):
    # TODO
    raise NotImplementedError()

def apply_clip(X, bounds):
    # TODO
    raise NotImplementedError()

In [ ]:
bounds = fit_clip_bounds(pd.DataFrame({'value': [0., 10., 20., 30., 40.]}), ['value'], 0.25, 0.75)
expected_bounds = pd.DataFrame({'lower': [10.], 'upper': [30.]}, index=['value'])
pd.testing.assert_frame_equal(bounds, expected_bounds)

sample = pd.DataFrame({'value': [-100., 200., np.nan]})
assert np.allclose(apply_clip(sample, bounds)['value'], [10., 30., np.nan], equal_nan=True)

In [ ]:
outlier_columns = ['age', 'hours_per_week', 'capital_gain']
iqr_bounds = fit_iqr_bounds(X_adult, outlier_columns)
display(iqr_bounds.assign(n_outliers=outlier_mask(X_adult, iqr_bounds).sum()))
clip_bounds = fit_clip_bounds(X_adult, ['hours_per_week', 'capital_gain'])
adult_clipped = apply_clip(X_adult, clip_bounds)
pd.DataFrame({'до': X_adult['capital_gain'].quantile([0.5, 0.95, 1.0]),
              'после': adult_clipped['capital_gain'].quantile([0.5, 0.95, 1.0])})
# Решите: какие значения стоит ограничивать, а какие лучше сохранить?

### [E. Редкие категории](https://contest.yandex.ru/contest/101948/problems/E/)

- `fit_rare_categories(X, columns, min_count=10)` → `{колонка: set категорий с числом появлений >= min_count}`. Пропуски не считайте; `min_count` — целое ≥1.
- `apply_rare_categories(X, state)` → копия X: в выбранных колонках редкие и новые категории заменены на `Other`, пропуски — на `__MISSING__`. Остальные колонки сохранены.


In [ ]:
import pandas as pd

def fit_rare_categories(X, columns, min_count=10):
    # TODO
    return {col: None for col in columns}

def apply_rare_categories(X, state):
    # TODO
    raise NotImplementedError()

In [ ]:
state = fit_rare_categories(pd.DataFrame({'city': ['a', 'a', 'b', None]}), ['city'], 2)
result = apply_rare_categories(pd.DataFrame({'city': ['a', 'b', 'new', None]}), state)
assert result['city'].tolist() == ['a', 'Other', 'Other', '__MISSING__']

In [ ]:
rare_state = fit_rare_categories(X_adult, ['occupation', 'native_country'], min_count=100)
adult_rare = apply_rare_categories(X_adult, rare_state)
pd.DataFrame({'до': X_adult[['occupation', 'native_country']].nunique(),
              'после': adult_rare[['occupation', 'native_country']].nunique()})
# TODO: сравните min_count=10 и 100; сколько информации теряется?

### [F. Новые признаки](https://contest.yandex.ru/contest/101948/problems/F/)

`add_adult_features(X)` → копия X с тремя добавленными колонками в этом порядке:

- `has_capital_gain` — флаг `capital_gain > 0`;
- `has_capital_loss` — флаг `capital_loss > 0`;
- `capital_gain_log` — `log1p(capital_gain)`.

Известные суммы неотрицательны. Если исходное значение пропущено, соответствующий новый признак — NaN.


In [ ]:
import numpy as np
import pandas as pd

def add_adult_features(X):
    # TODO
    raise NotImplementedError()

In [ ]:
sample = pd.DataFrame({'capital_gain': [0., 9., np.nan], 'capital_loss': [2., 0., np.nan]})
result = add_adult_features(sample)
assert np.allclose(result.has_capital_gain, [0., 1., np.nan], equal_nan=True)
assert np.allclose(result.capital_gain_log, [0., np.log(10), np.nan], equal_nan=True)

In [ ]:
adult_features = add_adult_features(X_adult)
adult_features[['capital_gain', 'capital_gain_log']].hist(bins=30, figsize=(9, 3))
plt.tight_layout()
plt.show()
# TODO: добавьте собственный признак в отдельной ячейке и объясните его смысл.

### [G. Frequency encoding](https://contest.yandex.ru/contest/101948/problems/G/)

- `fit_frequency(X, columns)` → `{колонка: {категория: доля в train}}`. Знаменатель — все строки; пропуск — категория `__MISSING__`. Для пустого train внутренние словари пустые.
- `apply_frequency(X, state)` → копия X с частотами вместо выбранных колонок. Неизвестная категория получает 0; пропуск — частоту `__MISSING__` или 0. Остальные колонки сохраните.


In [ ]:
import pandas as pd

def fit_frequency(X, columns):
    # TODO
    return {col: None for col in columns}

def apply_frequency(X, state):
    # TODO
    raise NotImplementedError()

In [ ]:
state = fit_frequency(pd.DataFrame({'city': ['a', 'a', 'b', None]}), ['city'])
result = apply_frequency(pd.DataFrame({'city': ['a', 'new', None]}), state)
assert np.allclose(result['city'], [0.5, 0., 0.25])

In [ ]:
frequency_state = fit_frequency(X_adult, ['occupation'])
adult_frequency = apply_frequency(X_adult, frequency_state)
pd.DataFrame({'категория': X_adult['occupation'], 'частота': adult_frequency['occupation']}).drop_duplicates().head(12)
# Найдите категории, которые после такого кодирования трудно различить.

### [H. Target encoding](https://contest.yandex.ru/contest/101948/problems/H/)

- `fit_target_encoding(categories, y)` → `{'mapping': категория → среднее y, 'global_mean': среднее y по train}`.
- `apply_target_encoding(categories, state)` → вещественная Series с исходным индексом и именем `target_encoding`. Неизвестным категориям — `global_mean`; пропуск считайте категорией `__MISSING__`.

Пустой train или несовпадающие индексы categories и y → `ValueError`. Пустая новая Series → пустой вещественный результат.


In [ ]:
import pandas as pd


def fit_target_encoding(categories, y):
    # TODO
    return {
        'mapping': None,
        'global_mean': None,
    }


def apply_target_encoding(categories, state):
    # TODO
    raise NotImplementedError

In [ ]:
categories = pd.Series(['a', 'a', 'b', None])
labels = pd.Series([0, 1, 0, 1])
state = fit_target_encoding(categories, labels)
new = pd.Series(['a', 'b', None, 'unseen'], index=[8, 2, 9, 4])
codes = apply_target_encoding(new, state)
assert np.allclose(codes, [0.5, 0., 1., 0.5])
assert codes.index.equals(new.index) and codes.name == 'target_encoding'

In [ ]:
occupation_encoding = fit_target_encoding(X_adult['occupation'], y_adult)
occupation_valid = apply_target_encoding(X_adult_valid['occupation'], occupation_encoding)
pd.DataFrame({'категория': X_adult_valid['occupation'], 'код по train': occupation_valid}).head(10)

### [I. Подготовка данных](https://contest.yandex.ru/contest/101948/problems/I/)

`fit_preprocessor(X)` → словарь из заготовки:

- `input_columns`, `numeric`, `categorical` — списки колонок в исходном порядке; числовые определяйте по dtype `number`;
- `medians` — Series медиан train, для полностью пустой колонки — 0;
- `scaler` — `StandardScaler`, обученный после заполнения числовых пропусков; если числовых колонок нет — None;
- `columns` — колонки после замены категориальных пропусков на `__MISSING__` и `pd.get_dummies(..., columns=categorical, dtype=float)` без удаления первой категории; числовые колонки тоже включите.

`transform_preprocessor(X, state)` → числовой DataFrame: заполните пропуски сохранёнными медианами, примените scaler и такое же one-hot-кодирование. Колонки и их порядок должны совпадать с `state['columns']`; отсутствующие заполните нулями, лишние уберите. Для пустого X верните пустую таблицу с этими колонками.

Train непустой. При преобразовании новых данных ничего заново не обучайте.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler


def fit_preprocessor(X):
    # TODO
    return {
        'input_columns': None,
        'numeric': None,
        'categorical': None,
        'medians': None,
        'scaler': None,
        'columns': None,
    }


def transform_preprocessor(X, state):
    # TODO
    raise NotImplementedError

In [ ]:
train = pd.DataFrame({'value': [1., 3., np.nan], 'city': ['a', 'b', np.nan]})
state = fit_preprocessor(train)
assert isinstance(state['scaler'], StandardScaler)
new = pd.DataFrame({'value': [2., np.nan], 'city': ['unknown', 'a']}, index=[8, 4])
prepared = transform_preprocessor(new, state)
assert np.allclose(prepared['value'], [0., 0.])
assert prepared.index.equals(new.index) and prepared.iloc[0].eq(0).all()

In [ ]:
preprocessing_state = fit_preprocessor(adult_features)
prepared_adult = transform_preprocessor(adult_features, preprocessing_state)
pd.Series({'строк': len(prepared_adult), 'признаков': prepared_adult.shape[1],
           'пропусков': prepared_adult.isna().sum().sum()})
# Это просмотр преобразования на train. Для оценки модели начинайте с исходного X_adult.

### [J. PCA](https://contest.yandex.ru/contest/101948/problems/J/)

- `fit_pca(X, n_components)` → `{'columns': список колонок X, 'pca': обученный PCA(n_components=n_components, svd_solver='full')}`.
- `transform_pca(X, state)` → DataFrame с исходным индексом и колонками `PC1`, …, `PCk`. Используйте сохранённый порядок входных колонок. Для пустого X верните пустую таблицу с нужными PC-колонками.

X уже числовой и без пропусков; дополнительно заполнять или масштабировать его не нужно. `n_components` — целое от 1 до `min(X.shape)`, иначе `ValueError`.


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA


def fit_pca(X, n_components):
    # TODO
    return {
        'columns': None,
        'pca': None,
    }


def transform_pca(X, state):
    # TODO
    raise NotImplementedError

In [ ]:
train = pd.DataFrame({'a': [1., 2., 3.], 'b': [2., 4., 6.]})
state = fit_pca(train, n_components=1)
new = pd.DataFrame({'b': [12.], 'a': [6.]}, index=[7])
reduced = transform_pca(new, state)
assert reduced.columns.tolist() == ['PC1'] and reduced.index.equals(new.index)
assert np.isclose(abs(reduced.loc[7, 'PC1']), 4 * np.sqrt(5))

In [ ]:
pca_state = fit_pca(prepared_adult, n_components=5)
adult_reduced = transform_pca(prepared_adult, pca_state)
pd.Series({'исходных признаков': prepared_adult.shape[1],
           'после PCA': adult_reduced.shape[1],
           'сохранённая доля дисперсии': pca_state['pca'].explained_variance_ratio_.sum()})

### [K. Итоговая модель](https://contest.yandex.ru/contest/101948/problems/K/)

- `fit_model(X_train, y_train)` → `{'preprocessing': настройки подготовки, 'model': обученная модель}`. Дополнительные ключи разрешены.
- `predict_proba(state, X)` → одномерный `numpy.ndarray` формы `(len(X),)` с вероятностями класса 1, конечными и в [0, 1]. Для пустого X — пустой вещественный массив. При прогнозе ничего заново не обучайте; результат строки не должен зависеть от остальных строк в вызове.

**Модель:** `LogisticRegression` или `RandomForestClassifier`, с `random_state=42`, без Pipeline и ансамбля моделей. Начните с `LogisticRegression`; ниже готовый baseline, использующий функции I.

**Разрешённые модули:** `numpy`, `pandas`, `sklearn.linear_model`, `sklearn.ensemble`, `sklearn.preprocessing`, `sklearn.impute`, `sklearn.neighbors`, `sklearn.decomposition`, `sklearn.feature_selection`.

В файл K включите все используемые помощники, в том числе функции I, если они нужны вашему решению. Проверяющая система обучает модель на выданном train; validation используйте для своих экспериментов.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression


def fit_model(X_train, y_train):
    # Сохраняем настройки подготовки, полученные только по train.
    preprocessing = fit_preprocessor(X_train)
    X_ready = transform_preprocessor(X_train, preprocessing)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_ready, y_train)
    return {
        'preprocessing': preprocessing,
        'model': model,
    }


def predict_proba(state, X_val):
    if len(X_val) == 0:
        return np.empty(0, dtype=float)
    # Применяем готовые настройки, ничего заново не обучаем.
    X_val_ready = transform_preprocessor(X_val, state['preprocessing'])
    return state['model'].predict_proba(X_val_ready)[:, 1]  # вероятность класса 1

In [ ]:
sample = pd.DataFrame({'age': [20,25,30,35,40,45], 'capital_gain': [0.,0.,2.,0.,10.,5.],
                       'capital_loss': [0.,1.,0.,0.,0.,2.], 'occupation': ['a','a','b','b','a','b']})
labels = pd.Series([0,0,0,1,1,1])
state = fit_model(sample, labels)
proba = predict_proba(state, sample.iloc[:2])
assert proba.shape == (2,)
assert np.isfinite(proba).all() and ((proba >= 0) & (proba <= 1)).all()

### Посчитаем качество

Обучите решение на train и получите вероятности для validation — ячейка ниже посчитает ROC-AUC. Для каждого нового варианта начинайте с исходного train, чтобы сравнение было понятным.

In [ ]:
model_state = fit_model(X_adult, y_adult)
valid_proba = predict_proba(model_state, X_adult_valid)
pd.Series({'validation_ROC_AUC': roc_auc_score(y_adult_valid, valid_proba)})

## Дополнительно: отбор признаков и PCA на Ames

Вернёмся к домам. Сравните LinearRegression на всех числовых признаках, после SelectKBest(k=6) и после PCA с шестью компонентами. Для PCA используйте функции из задачи J. Сначала заполните пропуски и масштабируйте данные — отдельно от PCA, обучая каждый шаг только на train.

Оценивайте на том же `X_ames_eval`. SelectKBest оставляет исходные признаки, а PCA создаёт их линейные комбинации. Что случилось с RMSE и MAE? Почему большая сохранённая доля дисперсии ещё не гарантирует хорошее предсказание цены?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Загрузим Ames отдельно: переменные семинарского ноутбука здесь не нужны.
ames_train = pd.read_csv('data/ames_train.csv')
ames_valid = pd.read_csv('data/ames_valid.csv')
X_ames, y_ames = ames_train.drop(columns='sale_price'), ames_train['sale_price']
X_ames_valid, y_ames_valid = ames_valid.drop(columns='sale_price'), ames_valid['sale_price']
numeric_columns = X_ames.select_dtypes('number').columns.tolist()
valid_complete = X_ames_valid[numeric_columns].notna().all(axis=1)
X_ames_eval = X_ames_valid.loc[valid_complete].copy()
y_ames_eval = y_ames_valid.loc[valid_complete]

In [ ]:
# TODO: сравните три варианта и сформулируйте вывод.
raise NotImplementedError